# 04 业务建议、ROI 测算与 A/B 实验设计

前三周回答了"发生了什么、为什么、谁能被预测"；本周回答**"所以该做什么、值多少钱、怎么验证"**。

每条建议的结构：证据（前序分析）→ 动作 → 预期收益（显式假设下的测算）→ A/B 验证方案（含样本量计算）。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "clean"

scores = pd.read_parquet(CLEAN / "churn_scores.parquet")
tx = pd.read_parquet(CLEAN / "transactions.parquet")
tx = tx[tx["has_customer"]].copy()
tx["Amount"] = tx["Quantity"] * tx["Price"]

T_MAX = tx["InvoiceDate"].max()
T_CUT = T_MAX - pd.Timedelta(days=90)

# 预测窗口内每位复购客户的实际消费（用于估算"留住一个人值多少钱"）
fut_spend = tx[tx["InvoiceDate"] > T_CUT].groupby("Customer_ID")["Amount"].sum()
scores = scores.merge(fut_spend.rename("spend_90d"), left_on="Customer_ID", right_index=True, how="left")
scores["spend_90d"] = scores["spend_90d"].fillna(0)

n = len(scores)
top10 = scores.nlargest(int(n * 0.10), "churn_score")
top20 = scores.nlargest(int(n * 0.20), "churn_score")
print(f"活跃客群 {n:,} 人 | top10% = {len(top10)} 人 | top20% = {len(top20)} 人")
print(f"复购客户 90 天消费：中位 £{(scores.loc[scores.repurchase == 1, 'spend_90d']).median():,.0f}，均值 £{(scores.loc[scores.repurchase == 1, 'spend_90d']).mean():,.0f}")

活跃客群 2,788 人 | top10% = 278 人 | top20% = 557 人
复购客户 90 天消费：中位 £641，均值 £1,592


## 建议 1：高风险客群挽留券（核心动作）

**证据链**：W2 发现增长全靠老客（活跃中 88.5% 为老客）→ W3 模型识别出高风险名单 → 本节算这条名单值多少钱、券怎么发不亏。

关键基线：**不干预时**，高风险客户自己回头的比例（历史实测）。

In [2]:
p0 = 1 - top20["repurchase"].mean()  # top20% 高风险名单的自然流失率
winback_base = top20["repurchase"].mean()  # 不发券也会回来的比例
value_kept = scores.loc[scores.repurchase == 1, "spend_90d"].median()  # 用中位数抗长尾

print(f"top20% 名单自然流失率 {p0:.1%}（即 {winback_base:.0%} 的人不发券也会回来——这正是必须做对照组的原因）")
print(f"留住一位复购客户的 90 天消费中位数：£{value_kept:,.0f}")

# ROI 测算（假设全部显式列出，面试时讲"假设怎么来的"比讲结果重要）
COUPON = 10          # 每人挽留券面额（英镑）
LIFT = 0.05          # 券带来的增量回头率（行业经验值 3-8pp，取中位）
N_TARGET = len(top20)
cost = N_TARGET * COUPON
incremental = N_TARGET * LIFT * value_kept
print(f"\n【ROI 测算】目标 {N_TARGET} 人，券面额 £{COUPON}，假设增量回头 +{LIFT:.0%}")
print(f"成本 £{cost:,.0f} / 预期增量收入 £{incremental:,.0f} / ROI = {incremental / cost:.1f}x")

top20% 名单自然流失率 82.6%（即 17% 的人不发券也会回来——这正是必须做对照组的原因）
留住一位复购客户的 90 天消费中位数：£641

【ROI 测算】目标 557 人，券面额 £10，假设增量回头 +5%
成本 £5,570 / 预期增量收入 £17,852 / ROI = 3.2x


## A/B 实验设计：多少样本才够下结论？

把高风险名单随机对半分成 对照组（不发券）/ 实验组（发券），主指标 = 90 天复购率。
用两比例检验的功效分析算每组所需样本（α=0.05 双侧，power=0.8，MDE=+8pp）。

In [3]:
from scipy.stats import norm

MDE = 0.08  # 最小可检测效应：回头率 +8pp
p_ctrl = winback_base
p_treat = p_ctrl + MDE

# 两比例检验样本量（双侧 α=0.05，power=0.8）——直接按公式实现：
# n = [z_α·√(2·p̄·(1-p̄)) + z_β·√(p₁(1-p₁)+p₀(1-p₀))]² / (p₁-p₀)²
z_a = norm.ppf(0.975)
z_b = norm.ppf(0.80)
p_bar = (p_ctrl + p_treat) / 2
n_per_group = int(np.ceil((z_a * np.sqrt(2 * p_bar * (1 - p_bar)) + z_b * np.sqrt(p_treat * (1 - p_treat) + p_ctrl * (1 - p_ctrl))) ** 2 / MDE**2))
need = 2 * n_per_group
print(f"对照组回头率基线 {p_ctrl:.0%}，MDE +{MDE:.0%} → 每组需 {n_per_group:,} 人，共 {need:,} 人")

k = 0.10
while int(n * k) < need:
    k += 0.05
final_list = int(n * k)
print(f"名单需扩至活跃客群 top {k:.0%}（{final_list} 人）才能满足统计功效")
print(f"若只投 top10%（{int(n * 0.10)} 人）< 所需 {need:,} 人：实验会「测不出真实效应」——宁可慢，不可白做")

对照组回头率基线 17%，MDE +8% → 每组需 412 人，共 824 人
名单需扩至活跃客群 top 30%（836 人）才能满足统计功效
若只投 top10%（278 人）< 所需 824 人：实验会「测不出真实效应」——宁可慢，不可白做


## 建议 2：新客承接优化（次月留存 21.2% 太低）

**证据**：W2——新客量已腰斩（-55%），次月留存仅 21.2%，每一个新客都更贵了，接不住就是双重浪费。

动作：首购后 D+7 发二单券 + 首购欢迎礼盒（英区电商标准做法），目标次月留存 +5pp。

In [4]:
NEW_2011 = 1508        # 2011 前 11 个月新客（W2 实测）
RET_LIFT = 0.05        # 目标：次月留存 21.2% -> 26.2%
order_value_new = float(np.median(tx.groupby("Invoice")["Amount"].sum()))  # 客单价中位数

saved = NEW_2011 * RET_LIFT
print(f"2011 年新客 {NEW_2011:,} 人 × 留存 +{RET_LIFT:.0%} = 多留住 {saved:.0f} 人/年")
print(f"按客单价中位数 £{order_value_new:,.0f}、保守估每人每年多 1.5 单：年增量 GMV ≈ £{saved * order_value_new * 1.5:,.0f}")

2011 年新客 1,508 人 × 留存 +5% = 多留住 75 人/年
按客单价中位数 £303、保守估每人每年多 1.5 单：年增量 GMV ≈ £34,274


## 建议 3：把"假流失"批发客户从券名单里摘出来

**证据**：W3 逻辑回归——客单价越高（OR 0.61）、资历越老（OR 0.60）越"像"流失，实际是批发客户天然低频。给他们发券是浪费，他们要的是定期补货提醒/大客户经理。

In [5]:
# 从 top20% 名单里识别高客单价客户（历史客单价位于全体前 25%）
hist = tx[tx["InvoiceDate"] <= T_CUT]
aov = hist.groupby("Customer_ID").apply(
    lambda g: g["Amount"].sum() / g["Invoice"].nunique(), include_groups=False
).rename("aov_hist")
top20 = top20.merge(aov, left_on="Customer_ID", right_index=True)
aov_thresh = aov.quantile(0.75)
wholesale = top20[top20["aov_hist"] >= aov_thresh]
coupon_list = top20[top20["aov_hist"] < aov_thresh]

print(f"top20% 名单 {len(top20)} 人中：高客单价批发型 {len(wholesale)} 人（{len(wholesale) / len(top20):.0%}）")
print(f"→ 券名单瘦身至 {len(coupon_list)} 人，批发客户转入「大客户定期补货」流程")
print(f"省下券成本 £{len(wholesale) * COUPON:,.0f}，且避免对错误人群营销损伤品牌")

top20% 名单 557 人中：高客单价批发型 120 人（22%）
→ 券名单瘦身至 437 人，批发客户转入「大客户定期补货」流程
省下券成本 £1,200，且避免对错误人群营销损伤品牌


## 结论（2026-11-05 执行）

**三条建议全部量化落地：**

1. **挽留券 ROI 3.2x**：向 top20% 高风险名单（557 人）发 £10 挽留券，假设增量回头率 +5pp，成本 £5,570 → 预期增量收入 £17,852。A/B 验证需 824 人（每组 412；α=0.05、power=0.8、MDE +8pp），名单须扩至活跃客群 top30%。
2. **新客承接值 £3.4 万/年**：2011 年 1,508 位新客 × 次月留存 +5pp = 多留住 75 人，按客单价中位 £303、人均年 1.5 单保守测算，年增量 GMV ≈ £34,274。
3. **假流失陷阱：省 £1,200 + 避免误伤**：557 人名单中 22%（120 人）是高客单价批发型低频客户——从券名单摘出（终版 437 人），转入「大客户定期补货」流程。

**最重要的方法论提醒**：top20% 名单中 17% 的人不发券也会回来——没有对照组的挽留活动会把「自然回头」记成「券的功劳」，ROI 全是幻觉。这也是为什么建议 1 必须以 A/B 而非全量投放的形式落地。
